# 🌿🏛️ ArcSpectra - Multispectral Vegetation Index Processor

Welcome to **ArcSpectra**, a high-performance Python tool designed for remote sensing archaeology and vegetation stress analysis. 
This notebook allows you to calculate and visualize optical spectral indices (such as NDVI, GNDVI, WDVI, NAVI, and more) from drone orthomosaics or satellite GeoTIFF rasters using the same underlying calculation engine as the ArcSpectra desktop GUI.

### Workflow Overview:
1. Define input and output file paths.
2. Configure band number mappings and select indices of interest.
3. Load index database formulas from the Geopera spectral registry.
4. Execute raster calculations dynamically inside elements-wise NumPy environments.
5. Plot and visualize the results for cropmark or soilmark detection.

### Step 1: Load Dependencies and Configure Settings
Set up paths and configure settings. The inputs are calibrated to target archaeological features like cropmarks.

In [ ]:
import os
import json
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Import the shared calculation engine
import processor

# 1. File Configuration
INPUT_FILE = "input/2024-05-26_Ortho.tif"
OUTPUT_DIR = "output/2024-05-26_Ortho"

# 2. Sensor Band Mapping (maps band name -> band number in GeoTIFF)
# Edit these to match your sensor (e.g. DJI, Sentinel-2, Landsat)
BAND_MAPPING = {
    "Red": 1,
    "Green": 2,
    "Blue": 3,
    "NIR": 4
}

# 3. Indices Selections (matching compiled_indices.json registry entries)
SELECTED_INDICES = ["NDVI", "GNDVI", "WDVI", "NAVI", "SAVI (Arch)"]

# 4. Soil line slope parameter (s) for WDVI
WDVI_SOIL_SLOPE = 1.0

# 5. Reflectance Scaling Mode: 'auto' or a numeric factor (e.g. 255.0, 65535.0)
SCALE_FACTOR = "auto"

### Step 2: Load Geopera Database and Parse Formulas
We read from the compiled Geopera Database to load standard or custom archaeological formulas.

In [ ]:
db_file = "compiled_indices.json"
if not os.path.exists(db_file):
    raise FileNotFoundError(f"Database file not found: {db_file}. Please run the GUI or build compiling script first.")

with open(db_file, "r", encoding="utf-8") as f:
    all_indices = json.load(f)

print(f"[✓] Loaded {len(all_indices)} spectral indices from Geopera Database.")

# Build formula dictionary mapping abbreviation -> formula
selected_formulas = {}
for index_def in all_indices:
    abbrev = index_def["abbrev"]
    if abbrev in SELECTED_INDICES:
        selected_formulas[abbrev] = index_def["formula"]

print(
    f"\n[✓] Mapped {len(selected_formulas)} formulas for active calculation:"
)
for ab, fm in selected_formulas.items():
    print(f"  - {ab}: {fm}")

### Step 3: Run Raster Index Calculation Pipeline
Executes the core raster processor which performs block-based matrix operations.

In [ ]:
# Verify input
if not os.path.exists(INPUT_FILE):
    print(f"[!] Warning: Input file '{INPUT_FILE}' does not exist yet. Please select an existing file.")
else:
    # Resolve auto scale factor value
    active_scale = 1.0
    if SCALE_FACTOR == "auto":
        with rasterio.open(INPUT_FILE) as src:
            sample = src.read(1, out_shape=(100, 100)).astype(np.float32)
            max_val = np.nanmax(sample)
            if max_val > 1.5:
                active_scale = 255.0 if max_val <= 255.0 else 65535.0
        print(f"Auto-detected scaling factor: {active_scale}")
    else:
        active_scale = float(SCALE_FACTOR)

    print("\nStarting calculation pipeline...")
    ortho_name, results, saved_files, plot_file = processor.process_raster_file(
        file_path=INPUT_FILE,
        output_dir=OUTPUT_DIR,
        bands_mapping=BAND_MAPPING,
        selected_indices_formulas=selected_formulas,
        wdvi_s=WDVI_SOIL_SLOPE,
        scale_factor=active_scale,
        log_callback=print
    )

### Step 4: Display Output Overview Plot
Renders the combined indices overview PNG saved in your output directory.

In [ ]:
if 'plot_file' in locals() and plot_file and os.path.exists(plot_file):
    print("Overview plot saved at:", plot_file)
    display(Image(filename=plot_file))
else:
    print("Overview plot not found. Please run Step 3 successfully first.")

### Step 5: High-Resolution Individual Index Inspection
Render an individual index map (e.g. NDVI or NAVI) with custom colormaps and extract metrics.

In [ ]:
# Select index to inspect (e.g. "NDVI", "NAVI", "GNDVI")
INSPECT_INDEX = "NDVI"

if 'results' in locals() and INSPECT_INDEX in results:
    data = results[INSPECT_INDEX]
    
    # Renders the index mapping using Red-Yellow-Green gradient
    plt.figure(figsize=(12, 10), dpi=150)
    plt.imshow(data, cmap="RdYlGn")
    plt.colorbar(label=f"{INSPECT_INDEX} Value")
    plt.title(f"ArcSpectra: High-Resolution {INSPECT_INDEX} Cropmark Analysis", fontsize=14, fontweight="bold")
    plt.xlabel("X (pixels)")
    plt.ylabel("Y (pixels)")
    plt.grid(False)
    plt.show()
    
    # Calculate stats
    valid = data[~np.isnan(data) & ~np.isinf(data)]
    if len(valid) > 0:
        print(f"\nSummary statistics for {INSPECT_INDEX}:")
        print(f"  - Minimum:    {np.nanmin(valid):.4f}")
        print(f"  - Maximum:    {np.nanmax(valid):.4f}")
        print(f"  - Mean:       {np.nanmean(valid):.4f}")
        print(f"  - Std Dev:    {np.nanstd(valid):.4f}")
else:
    print(f"Index {INSPECT_INDEX} is not available in results.")